In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes one argument:
- `n` is a natural number.

It returns the list $[0, 1, \cdots, n-1]$.

In [ ]:
let range (n : int) : int list = List.init n Fun.id

In [ ]:
range 5

# Sudoku

The sudoku we want to solve is shown here:
    <img src="sudoku.png" width="40%">
I have taken it from https://sudoku.zeit.de/sudoku-hoellisch.

The function `create_puzzle` takes the argument `()`.  It returns a representation of this puzzle as a list of rows, where every row is a list of 9 cells.  A cell that contains the digit `d` is represented as `Some d`, an empty cell is represented as `None`.  In order to make the puzzle easy to read, every row is written as a string that contains a `'*'` for every empty cell.

In [ ]:
let create_puzzle () : int option list list =
  let rows = ["*39*****7";
              "***7**492";
              "****65*83";
              "***6*327*";
              "****4*8**";
              "56*******";
              "**52*9**1";
              "*21****4*";
              "7*****5**"] in
  let cell (ch : char) : int option =
    if ch = '*' then None else Some (Char.code ch - Char.code '0')
  in
  List.map (fun row -> List.map cell (List.of_seq (String.to_seq row))) rows

We are going to solve this puzzle with the help of the constraint solver `Z3`.

In [ ]:
import_file "../z3_helpers.ml"

The function `constraints_from_puzzle` takes one argument:
* `variables` is a matrix of `Z3` variables.

   For `row`$\in \{0,\cdots,8\}$ and `col`$\in \{0,\cdots,8\}$ the variable `variables.(row).(col)` specifies the number that is placed in the specified row and column.  The matrix is represented as an array of arrays, so that its entries can be accessed via their indices.

It returns a list of constraints specifying that the variables corresponding to numbers that are already set in the given Sudoku take the specified values.

Replace every occurrence of `failwith "your code here"` with your code.  The function `failwith` raises an exception.  As it can be used in place of an expression of any type, it serves as a placeholder.

In [ ]:
let constraints_from_puzzle (variables : Z3.Expr.expr array array) : Z3.Expr.expr list =
  failwith "your code here"

The matrix `variables` contains the variables `V11`, $\cdots$, `V99`.

In [ ]:
let variables : Z3.Expr.expr array array =
  Array.init 9 (fun row -> Array.init 9 (fun col -> int_const (Printf.sprintf "V%d%d" (row + 1) (col + 1))))

The function `all_constraints` takes one argument:
- `variables` is the matrix of variables.

It returns the list of all constraints that encode the given Sudoku:
- the constraints given by the puzzle,
- the constraints stating that the numbers in every row, every column, and every 3x3 square are different,
- the constraints stating that every number is between 1 and 9.

`Array.to_list` converts an array into a list.

In [ ]:
let all_constraints (variables : Z3.Expr.expr array array) : Z3.Expr.expr list =
  let constraints     = constraints_from_puzzle variables in
  let row_constraints = List.map (fun row -> distinct (Array.to_list variables.(row))) (range 9) in
  let col_constraints    : Z3.Expr.expr list = failwith "your code here" in
  let square_constraints : Z3.Expr.expr list = failwith "your code here" in
  let bound_constraints  : Z3.Expr.expr list = failwith "your code here" in
  constraints @ row_constraints @ col_constraints @ square_constraints @ bound_constraints

I have got 217 constraints.

In [ ]:
List.length (all_constraints variables)

The function `solve` takes the argument `()`.  It computes a solution to the given problem.  The result is `Some solution`, where `solution` is a map that maps the names of the variables to their values, or `None` if no solution can be found.  The function `Z3.Solver.check` returns one of the constructors `SATISFIABLE`, `UNSATISFIABLE`, and `UNKNOWN` of the type `Z3.Solver.status`.

In [ ]:
let solve () : (string, int) RMap.t option =
  let variables =
    Array.init 9 (fun row -> Array.init 9 (fun col -> int_const (Printf.sprintf "V%d%d" (row + 1) (col + 1))))
  in
  let s = new_solver () in
  Z3.Solver.add s (all_constraints variables);
  match Z3.Solver.check s [] with
  | Z3.Solver.SATISFIABLE ->
      let m = solver_model s in
      Some (RMap.of_list
              (List.concat_map
                 (fun row ->
                    List.map
                      (fun col -> (Printf.sprintf "V%d%d" (row + 1) (col + 1)), eval_int m variables.(row).(col))
                      (range 9))
                 (range 9)))
  | Z3.Solver.UNSATISFIABLE ->
      print_endline "The problem is not solvable.";
      None
  | Z3.Solver.UNKNOWN ->
      print_endline "Z3 cannot determine whether the problem is solvable.";
      None

In [ ]:
let solution = time "Solver Duration" solve

## Graphical Representation

The function `show_solution` takes two arguments:
- `solution` is the result of `solve ()`,
- `width` is the width of the picture.  It is an optional argument with the default value `"50%"`.

It displays the solution on a 9x9 grid as an SVG picture.  The digits given by the puzzle are shown in bold face.

In [ ]:
#require "jupyter.notebook";;

In [ ]:
let show_solution ?(width = "50%") (solution : (string, int) RMap.t option) : unit =
  match solution with
  | None -> print_endline "No solution to display."
  | Some solution ->
      let sudoku     = create_puzzle () in
      let cell_size  = 50 in
      let total_size = 9 * cell_size in
      let html = Buffer.create 8192 in
      Buffer.add_string html
        (Printf.sprintf "<svg width=\"%s\" viewBox=\"-5 -5 %d %d\" xmlns=\"http://www.w3.org/2000/svg\" style=\"font-family: sans-serif;\">"
           width (total_size + 10) (total_size + 10));
      List.iteri
        (fun r cells ->
           List.iteri
             (fun c cell ->
                let x = c * cell_size and y = r * cell_size in
                let (value, bg, color, weight) =
                  match cell with
                  | Some d -> (d, "#f0f0f0", "#000000", "bold")
                  | None   -> (RMap.find (Printf.sprintf "V%d%d" (r + 1) (c + 1)) solution, "#ffffff", "#1a73e8", "normal")
                in
                Buffer.add_string html
                  (Printf.sprintf "<rect x=\"%d\" y=\"%d\" width=\"%d\" height=\"%d\" fill=\"%s\" stroke=\"#cccccc\" stroke-width=\"1\" />"
                     x y cell_size cell_size bg);
                Buffer.add_string html
                  (Printf.sprintf "<text x=\"%d\" y=\"%d\" text-anchor=\"middle\" font-size=\"22\" fill=\"%s\" font-weight=\"%s\">%d</text>"
                     (x + cell_size / 2) (y + cell_size / 2 + 7) color weight value))
             cells)
        sudoku;
      List.iter
        (fun i ->
           let pos = i * cell_size in
           Buffer.add_string html
             (Printf.sprintf "<line x1=\"0\" y1=\"%d\" x2=\"%d\" y2=\"%d\" stroke=\"#333333\" stroke-width=\"3\" stroke-linecap=\"square\" /><line x1=\"%d\" y1=\"0\" x2=\"%d\" y2=\"%d\" stroke=\"#333333\" stroke-width=\"3\" stroke-linecap=\"square\" />"
                pos total_size pos pos pos total_size))
        [0; 3; 6; 9];
      Buffer.add_string html "</svg>";
      ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

In [ ]:
show_solution solution